# 04.03 · Cruce con registros previos

Cada operación de la agenda se busca en lo que la Dirección ya registró antes, para traer lo
que no sale del documento y no se debe redactar de nuevo:

| Fuente | Qué se trae |
|---|---|
| Input CCI del paso 2 (`idos_concepto.csv`) | Evidencia, factores (recomendaciones), decisión y concepto de priorización |
| Input CCI histórico (`data/raw/Libros excel/Input CCI.xlsx`, hoja `IDOs-Soberanos`) | Lo mismo, para IDOs registrados antes del flujo automático |
| Preparatorias anteriores (`Preparación CCI y CFE - VPE y GPID.xlsx`, hojas `CCI-2026` y `CCI-2025`) | Fecha, aprendizajes, recomendaciones, alertas y preguntas de la última vez que se vio |
| Insumos Evid+Recom (`reports/` y `data/processed/3. Insumo Evid+Recom/`) | Nombre del Word, para sugerir la Retroalimentación DADMI |

El cruce es por nombre y país, sin API. `ajustes_cruce` permite corregirlo a mano. La entrada
es `fichas.csv` del `04.02` y la salida
`data/interim/4. Preparatoria CCI/04.03_cruce_registros/cruce.csv`.

## 1. LOAD LIBRARIES

pandas lee los Excel y las tablas; el cruce usa solo la biblioteca estándar. Las rutas se toman
de `caf_input_cci_preparatoria/config.py`.

In [ ]:
from pathlib import Path
import re
import sys
import unicodedata
import warnings

from IPython.display import display
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))  # para importar el paquete caf_input_cci_preparatoria

from caf_input_cci_preparatoria import config

warnings.filterwarnings("ignore", module="openpyxl")  # imágenes WMF del Excel de Preparación

## 2. LOAD DATA

### Rutas y ajustes manuales

`ajustes_cruce` corrige el cruce de un documento, identificado por su nombre de archivo. Para
cada fuente (`input`, `preparatoria`, `insumo`) se da el nombre exacto del registro que
corresponde, o `""` para dejarlo sin cruce. Ejemplo:

```python
ajustes_cruce = {
    "02. RC Y DEC BNB Revision Octubre 2026 VF.pdf": {"preparatoria": ""},
}
```

In [ ]:
carpeta_paso = config.INTERIM / "4. Preparatoria CCI"
archivo_entrada = carpeta_paso / "04.02_ficha_documentos" / "fichas.csv"
archivo_salida = carpeta_paso / "04.03_cruce_registros" / "cruce.csv"

archivo_paso2 = (
    config.INTERIM / "2. IDOs - Input CCI" / "02.04_concepto_priorizacion" / "idos_concepto.csv"
)
archivo_input_historico = config.RAW_LIBROS / "Input CCI.xlsx"
archivo_preparacion = config.RAW_LIBROS / "Preparación CCI y CFE - VPE y GPID.xlsx"
carpetas_insumos = [config.REPORTS, config.PROCESSED / "3. Insumo Evid+Recom"]

ajustes_cruce = {}

### Fichas del 04.02

Una fila por documento de la agenda, con el nombre y el país que leyó Azure.

In [ ]:
def leer(ruta):
    return pd.read_csv(ruta, encoding="utf-8-sig", dtype=str, keep_default_na=False)


fichas = leer(archivo_entrada)
fichas.index = pd.RangeIndex(1, len(fichas) + 1)
print(f"{len(fichas)} documentos.")

### Registros previos

Cada fuente se lleva a las mismas columnas: `nombre`, `pais`, `fecha` y los textos que aporta.
Se descartan las filas sin nombre. En el Input CCI, el paso 2 va primero: si un IDO está en las
dos fuentes, se usa la versión del paso 2.

In [ ]:
def limpiar(valor):
    texto = "" if pd.isna(valor) else str(valor)
    return texto.replace("_x000D_", "").replace("\xa0", " ").strip()


def fecha_iso(valor):
    fecha = pd.to_datetime(limpiar(valor), errors="coerce")
    return "" if pd.isna(fecha) else fecha.date().isoformat()


paso2 = leer(archivo_paso2)
input_paso2 = pd.DataFrame(
    {
        "origen": "Input CCI (paso 2)",
        "nombre": paso2["operacion"].map(limpiar),
        "pais": paso2["pais"].map(limpiar),
        "fecha": paso2["fecha_registro"].map(fecha_iso),
        "evidencia": paso2["evidencia"],
        "recomendaciones": paso2["factores_impacto"],
        "decision_priorizacion": paso2["decision_sugerida"],
        "concepto_priorizacion": paso2["concepto"],
        "archivo_ido": paso2["archivo"],
    }
)

historico = pd.read_excel(archivo_input_historico, sheet_name="IDOs-Soberanos", dtype=str)
input_historico = pd.DataFrame(
    {
        "origen": "Input CCI histórico",
        "nombre": historico["Nombre operación"].map(limpiar),
        "pais": historico["País"].map(limpiar),
        "fecha": historico["Fecha IDO"].map(fecha_iso),
        "evidencia": historico["Evidencia"].map(limpiar),
        "recomendaciones": historico[
            "Factores para maximizar el impacto (lecciones aprendidas)"
        ].map(limpiar),
        "decision_priorizacion": historico["Decisión priorización"].map(limpiar),
        "concepto_priorizacion": historico["Concepto de priorización para evaluación"].map(
            limpiar
        ),
        "archivo_ido": "",
    }
)
registros_input = pd.concat([input_paso2, input_historico], ignore_index=True)
registros_input = registros_input[registros_input["nombre"].ne("")].reset_index(drop=True)
print(f"Input CCI: {len(input_paso2)} IDOs del paso 2 y {len(input_historico)} filas históricas.")

In [ ]:
hojas = pd.read_excel(
    archivo_preparacion, sheet_name=["CCI-2026", "CCI-2025"], header=2, dtype=str
)
columnas_2025 = {
    "Monto": "Monto + Financiamiento verde",
    "Lecciones Aprendidas": "Recomendaciones 🟢 🟡 🔴 ⚪ ",
    "Modalidad": "Producto",
}
partes = []
for nombre_hoja, hoja in hojas.items():
    hoja = hoja.rename(columns=columnas_2025)
    partes.append(
        pd.DataFrame(
            {
                "origen": f"Preparatoria ({nombre_hoja})",
                "nombre": hoja["Nombre operación"].map(limpiar),
                "pais": hoja["País"].map(limpiar),
                "fecha": hoja["Fecha"].map(fecha_iso),
                "tipo_documento": hoja["Tipo de documento"].map(limpiar),
                "aprendizajes": hoja["Aprendizajes de Evidencia"].map(limpiar),
                "recomendaciones": hoja["Recomendaciones 🟢 🟡 🔴 ⚪ "].map(limpiar),
                "alertas": hoja["Alertas"].map(limpiar),
                "preguntas": hoja["Preguntas y comentarios sugeridos"].map(limpiar),
            }
        )
    )
registros_preparatoria = pd.concat(partes, ignore_index=True)
registros_preparatoria = registros_preparatoria[
    registros_preparatoria["nombre"].ne("")
].reset_index(drop=True)
print(f"Preparatorias anteriores: {len(registros_preparatoria)} filas (CCI-2026 y CCI-2025).")

In [ ]:
insumos = sorted(
    {
        archivo.name
        for carpeta in carpetas_insumos
        for archivo in carpeta.glob("*Evid*Recom*.docx")
        if not archivo.name.startswith("~$")
    }
)
registros_insumo = pd.DataFrame(
    {
        "origen": "Insumo Evid+Recom",
        "nombre": [re.sub(r"^.*Evid\+?Recom\s*-\s*", "", Path(n).stem) for n in insumos],
        "pais": "",
        "archivo_insumo": insumos,
    }
)
print(f"Insumos Evid+Recom: {len(registros_insumo)} Word.")

## 3. PROCESS DATA

### Comparación de nombres

Los nombres se comparan por sus palabras distintivas: sin tildes, sin números y sin palabras
genéricas (programa, proyecto, banco, línea de crédito, inversión patrimonial, fondo, nombres de
países…). Un registro
corresponde a la operación si:

- el país coincide (o falta en alguno de los dos), y
- comparten al menos dos palabras distintivas y estas son al menos el 60% de las del nombre
  más corto, o ambos nombres se reducen a la misma palabra (por ejemplo, "FIE").

Se guarda el mejor candidato de cada fuente con su puntaje. Si no cumple la regla pero comparte
al menos dos palabras y el 40% de las del nombre más corto, queda como "posible" para revisar;
si no, la fuente queda sin cruce.

In [ ]:
GENERICAS = {
    "programa",
    "proyecto",
    "prestamo",
    "linea",
    "lineas",
    "credito",
    "creditos",
    "revolvente",
    "comprometida",
    "no",
    "banco",
    "bank",
    "s",
    "sa",
    "sab",
    "ltda",
    "fase",
    "componente",
    "operacion",
    "apoyo",
    "financiamiento",
    "favor",
    "para",
    "del",
    "de",
    "la",
    "las",
    "los",
    "el",
    "en",
    "y",
    "e",
    "con",
    "por",
    "al",
    "a",
    "o",
    "ido",
    "incorporacion",
    "incoportacion",
    "solicitud",
    "solictud",
    "renovacion",
    "revision",
    "republica",
    "estado",
    "plurinacional",
    "oriental",
    "bolivariana",
    "federativa",
    "cooperativa",
    "ifd",
    "inversion",
    "patrimonial",
    "fondo",
    "fund",
}


def normalizar(texto):
    texto = unicodedata.normalize("NFKD", str(texto or ""))
    texto = "".join(c for c in texto if not unicodedata.combining(c)).casefold()
    return re.sub(r"[^a-z0-9]+", " ", texto).strip()


PAISES = {
    normalizar(pais)
    for pais in [
        "Argentina",
        "Barbados",
        "Bolivia",
        "Brasil",
        "Chile",
        "Colombia",
        "Costa Rica",
        "Ecuador",
        "El Salvador",
        "España",
        "Guatemala",
        "Honduras",
        "Jamaica",
        "México",
        "Multinacional",
        "Panamá",
        "Paraguay",
        "Perú",
        "Portugal",
        "República Dominicana",
        "Trinidad y Tobago",
        "Uruguay",
        "Venezuela",
    ]
}


def pais_base(texto):
    texto = normalizar(texto)
    return next((pais for pais in PAISES if pais in texto), texto)


def palabras(nombre):
    excluidas = GENERICAS | {p for pais in PAISES for p in pais.split()}
    return {
        palabra
        for palabra in normalizar(nombre).split()
        if len(palabra) > 1 and not palabra.isdigit() and palabra not in excluidas
    }


def puntaje(palabras_a, palabras_b):
    comunes = palabras_a & palabras_b
    if not comunes:
        return 0.0, 0
    return len(comunes) / min(len(palabras_a), len(palabras_b)), len(comunes)


def corresponde(valor, comunes, palabras_a, palabras_b):
    if comunes >= 2 and valor >= 0.6:
        return True
    return comunes == 1 and len(palabras_a) == 1 and len(palabras_b) == 1


def es_posible(valor, comunes):
    return comunes >= 2 and valor >= 0.4

In [ ]:
def mejor_registro(ficha, registros, preferir_reciente=False):
    """Mejor candidato de una fuente: (fila o None, puntaje, estado)."""
    palabras_ficha = palabras(ficha["nombre_operacion"])
    pais_ficha = pais_base(ficha["pais"])
    candidatos = []
    for posicion, registro in registros.iterrows():
        pais_registro = pais_base(registro["pais"])
        if pais_ficha and pais_registro and pais_ficha != pais_registro:
            continue
        palabras_registro = palabras(registro["nombre"])
        valor, comunes = puntaje(palabras_ficha, palabras_registro)
        acepta = corresponde(valor, comunes, palabras_ficha, palabras_registro)
        if acepta or es_posible(valor, comunes):
            orden = (registro["fecha"], valor) if preferir_reciente else (valor, comunes)
            candidatos.append((acepta, *orden, -posicion, valor))
    if not candidatos:
        return None, 0.0, "sin cruce"
    mejor = max(candidatos)
    acepta, posicion, valor = mejor[0], -mejor[-2], mejor[-1]
    return registros.loc[posicion], round(valor, 2), "cruce" if acepta else "posible"

### Cruce de cada documento

Para cada fuente se toma el mejor candidato; en las preparatorias, entre los que corresponden,
el más reciente que sea anterior a la sesión (no el de la misma sesión). Después se aplican los
`ajustes_cruce`.

In [ ]:
def registro_por_nombre(registros, nombre):
    encontrados = registros[registros["nombre"].eq(nombre)]
    if encontrados.empty:
        raise ValueError(f"ajustes_cruce: no hay un registro llamado {nombre!r}")
    return encontrados.iloc[-1]


def cruzar(ficha):
    fila = {}
    anteriores = registros_preparatoria[registros_preparatoria["fecha"] < ficha["fecha_sesion"]]
    fuentes = {
        "input": (registros_input, False),
        "preparatoria": (anteriores, True),
        "insumo": (registros_insumo, False),
    }
    ajustes = ajustes_cruce.get(ficha["archivo"], {})
    for fuente, (registros, reciente) in fuentes.items():
        if fuente in ajustes:
            nombre = ajustes[fuente]
            registro = registro_por_nombre(registros, nombre) if nombre else None
            valor, estado = (1.0, "ajuste manual") if nombre else (0.0, "sin cruce (ajuste)")
        else:
            registro, valor, estado = mejor_registro(ficha, registros, reciente)
        fila[f"{fuente}_estado"] = estado
        fila[f"{fuente}_puntaje"] = valor
        for campo, contenido in (registro if registro is not None else {}).items():
            fila[f"{fuente}_{campo}"] = contenido
    return fila


cruces = pd.DataFrame([cruzar(ficha) for ficha in fichas.to_dict("records")], index=fichas.index)
for fuente, registros in [
    ("input", registros_input),
    ("preparatoria", registros_preparatoria),
    ("insumo", registros_insumo),
]:
    for campo in registros.columns:  # columnas fijas aunque ningún documento cruce
        if f"{fuente}_{campo}" not in cruces:
            cruces[f"{fuente}_{campo}"] = ""
cruces = cruces.drop(columns=["insumo_pais"]).fillna("")

### Revisión del cruce

Para cada documento, el registro encontrado en cada fuente y su estado. Los "posible" se revisan
y, si corresponden, se confirman en `ajustes_cruce`.

In [ ]:
vista = pd.DataFrame(
    {
        "documento": fichas["archivo"].str[:45],
        "operación": fichas["nombre_operacion"].str[:55],
        **{
            fuente: cruces[f"{fuente}_estado"]
            + ": "
            + cruces[f"{fuente}_nombre"].str[:50]
            + " "
            + cruces.get(f"{fuente}_fecha", pd.Series("", index=cruces.index))
            for fuente in ["input", "preparatoria", "insumo"]
        },
    }
)
with pd.option_context("display.max_rows", None, "display.max_colwidth", 80):
    display(vista)

## 4. SAVE DATA

`cruce.csv` tiene las columnas de `fichas.csv` más, para cada fuente (`input_`, `preparatoria_`,
`insumo_`), el estado (`cruce`, `posible`, `sin cruce`, `ajuste manual`), el puntaje y los
campos del registro encontrado. Los registros "posible" se guardan pero el `04.04` y el `04.05`
solo usan los de estado `cruce` o `ajuste manual`.

In [ ]:
tabla = fichas.join(cruces)
archivo_salida.parent.mkdir(parents=True, exist_ok=True)
tabla.to_csv(archivo_salida, index=False, encoding="utf-8-sig")
print(f"{len(tabla)} documentos guardados en {archivo_salida.relative_to(ROOT)}")
for fuente in ["input", "preparatoria", "insumo"]:
    print(f"{fuente}: {cruces[f'{fuente}_estado'].value_counts().to_dict()}")